# Code-Graph Tier-0 Ontology — Design Spec

> **Status:** living design spec · **Crate:** `spur-graph` · **Live artifact:** `a66543657…` (indexed HEAD `b16fe530`)
> **Companion:** `docs/superpowers/specs/2026-06-04-code-graph-ontology-tier0-design.ipynb`

## What "Tier 0" means

SPUR's code graph is a **knowledge graph of source code**: every fact is a **triple** — a *subject* symbol, a *predicate* relation, an *object* symbol. Tier 0 is the **T-Box** (terminology box): the *schema of the schema*. It does not describe any one repository; it describes **what a well-formed fact is allowed to look like** for *every* repository SPUR ingests.

If Tier 1+ is "what the graph says," Tier 0 is "what the graph is *permitted* to say." It is the contract the extractor, the resolver, and every downstream consumer agree on.

## The five T-Box pieces

| # | Piece | One-line definition | Status |
|---|---|---|---|
| 1 | **Classes** (`NodeKind`) | The node taxonomy — what kinds of things can be a subject/object | ✅ have it |
| 2 | **Predicate domain/range signatures** | For each predicate, the allowed subject kinds (domain) and object kinds (range) | ⚠️ partial — declared in code as resolver guards, not as a first-class table; **violated in the live graph (P0)** |
| 3 | **Predicate sub-axes** | Refinements within a predicate (`edge_kind`, `bind_method`, `confidence`) | ⚠️ partial |
| 4 | **Inverses / cardinality / transitivity** | Algebraic properties of each predicate (`calls⁻¹ = called_by`, `contains` transitive, …) | ❌ missing (P2) |
| 5 | **Versioned realization contract** | A per-language matrix of which predicates are realized + a gate that prevents silent drift | ⚠️ partial — matrix exists, gate enforces only *definitions* (P1b); resolver-version not in the cache key (P0) |

This notebook walks each piece in detail, **grounded with live queries against the real graph** so the design is reproducible rather than asserted.

In [2]:
# --- Live evidence harness: read-only connection to the SPUR analyst graph artifact ---
# Every "evidence" cell below queries this same DuckDB artifact the `code_*` MCP tools use.
import duckdb, pandas as pd

ANALYST_DB = "/Volumes/Projects/spur/.spur/analyst.duckdb"
con = duckdb.connect(ANALYST_DB, read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()

# Sanity-check freshness + scale (Tier-0 reasoning is only valid against a known artifact)
q("""SELECT graph_content_hash[1:12] AS artifact, manifest_version[1:12] AS manifest,
            node_count, resolved_edge_count
     FROM _meta""")

,artifact,manifest,node_count,resolved_edge_count
0,a66543657ca6,a5d57285053f,49890,99592


## 1. The triple model — Subject · Predicate · Object

Every edge in the graph is a directed triple:

```
(subject: GraphNode)  --[ predicate: RelationKind ]-->  (object: GraphNode)
        ↑ a symbol             ↑ the relation                  ↑ a symbol (or an
     (the "domain" end)                                         unresolved label)
```

- **Subject / Object** are `GraphNode`s, each carrying a `NodeKind` (its *class*) and a `stable_symbol_id` (identity that survives rebuilds).
- **Predicate** is one `RelationKind` variant. After the `Uses` removal there are **10**: `Imports, Calls, Constructs, Contains, Implements, Defines, References, Extends, Links, Touches`.
- Every edge also carries **provenance** so consumers can reason about trust:
  - `confidence_score: f64` — how sure the resolver is (1.0 = exact/qualified bind, 0.5 = bare singleton guess, …).
  - `bind_method: &str` — *how* it resolved (`"fqn"`, `"scope_match"`, `"singleton"`, `"same_file_duplicate"`, `""` = plain/unattributed).
  - `edge_kind` — a structural sub-type (`calls`, `calls_dyn`, `references_hof`, `references_other`).

**Why provenance is Tier-0, not an afterthought:** a knowledge graph that cannot distinguish "I parsed this `fqn` exactly" from "I guessed the only same-named symbol" cannot be safely queried for impact analysis. The `(confidence_score, bind_method)` pair is the T-Box's honesty channel — and, as we'll see in Piece 2, the *absence* of a `bind_method` is itself a diagnostic signal.

Two facts can therefore disagree on confidence about the *same* (subject, predicate, object) — Tier-0 keeps both rather than collapsing them, deferring the merge to Tier 2 (fusion) and Tier 3 (inference).

## 2. Piece 1 — Classes (`NodeKind`) ✅

The **class taxonomy** is the set of node kinds a subject or object may be. It is the one Tier-0 piece SPUR fully has today. Defined as the `NodeKind` enum in `crates/spur-graph/src/schema.rs` and realized per language by each `definition_kind_map` in `crates/spur-graph/src/extract/languages.rs`.

The shared definition vocabulary (`@definition.<kind>` capture → `NodeKind`):

| Capture | NodeKind | | Capture | NodeKind |
|---|---|---|---|---|
| `@definition.module` | `Module` | | `@definition.trait` | `Trait` |
| `@definition.function` | `Function` | | `@definition.type_alias` | `TypeAlias` |
| `@definition.method` | `Method` | | `@definition.macro` | `Macro` |
| `@definition.class` | `Class` | | `@definition.field` | `Field` |
| `@definition.interface` | `Interface` | | `@definition.constant` | `Constant` |
| `@definition.struct` | `Struct` | | `@definition.section` | `Section` |
| `@definition.enum` | `Enum` | | `@definition.enum_variant` | `EnumVariant` |
| `@definition.impl` | `Impl` | | *(plus `File`, the per-file root)* | `File` |

**Design rules that make this Tier-0 (not just an enum):**
1. Every `@definition.*` capture in any query **must** have a matching `definition_kind_map` key (gate-enforced).
2. Every `NodeKind` used by a map **must** have an explicit `symbol_kind()` string — no `"symbol"` fallback.
3. A language realizes a class by adding the capture; absence is recorded as `-` (not modelled) or `TODO` (gap) in the coverage matrix, never left implicit.

The live class distribution across the whole worktree: ▼

In [3]:
# Piece 1 evidence — the live class (NodeKind) inventory.
# These are the only kinds a triple's subject or object can be.
q("""SELECT symbol_kind AS node_kind, count(*) AS n
     FROM nodes
     GROUP BY 1 ORDER BY n DESC""")

,node_kind,n
0,section,18154
1,function,12149
2,field,7015
3,method,4709
4,enum_variant,1990
5,struct,1535
6,constant,1177
7,impl,1162
8,module,1139
9,enum,455


## 3. Piece 2 — Predicate domain/range signatures ⚠️

A predicate is only well-formed if its endpoints are the *right kinds of thing*. `implements` from a function to a markdown section is nonsense even if both symbols exist. Tier-0 declares, per predicate, a **domain** (allowed subject kinds) and a **range** (allowed object kinds).

### The intended signature table

| Predicate | Domain (subject) | Range (object) | Realized as |
|---|---|---|---|
| `Implements` | Struct, Class, Enum | **Trait, Interface** | resolver range guard ✅ |
| `Extends` | Trait, Interface, Class | **Trait, Interface, Class** | resolver range guard ✅ |
| `Calls` | Function, Method | Function, Method | callable filter — but **not terminal** (P1a) |
| `Constructs` | Function, Method | Struct, EnumVariant, Class | resolver reclassification ✅ |
| `Contains` | Module, File, Class, … | any child symbol | structural ✅ |
| `Imports` | File, Module | Module, File, symbol | resolver candidate filter ✅ |
| `References` (HOF) | Function, Method | Function, Method | allowlist-gated ✅ |

Today this lives **only as imperative guards** inside the resolver (`relational_target_kinds()` in `extract/tree_sitter.rs`), not as a first-class, queryable signature table. That is Tier-0 piece-2's *partial* status: the constraint exists in code but is neither declared as data nor enforced as an invariant on the *persisted* graph.

### ⚠️ The live graph VIOLATES this signature (P0)

The range guard for `implements`/`extends` is correct in the current resolver — yet the live artifact still contains out-of-range edges. The query below is the empirical proof. Watch the `extends → enum_variant` / `extends → section` rows, all with an **empty `bind_method`** and **`conf = 0.5`**: ▼

In [4]:
# Piece 2 evidence — domain/RANGE for the relational predicates.
# Expectation (Tier-0):  implements -> {trait, interface}   extends -> {trait, interface, class}
# Reality: out-of-range enum_variant / section rows survive in the persisted graph.
df = q("""
  SELECT e.relation,
         dst.symbol_kind        AS target_kind,
         e.bind_method,
         round(avg(e.confidence_score), 3) AS conf,
         count(*)               AS n
  FROM edges e
  JOIN nodes dst ON dst.stable_symbol_id = e.target_stable_id
  WHERE e.relation IN ('extends', 'implements')
  GROUP BY 1, 2, 3
  ORDER BY 1, n DESC
""")

in_range = {'extends': {'trait','interface','class'}, 'implements': {'trait','interface'}}
df['in_range'] = [tk in in_range[r] for r, tk in zip(df.relation, df.target_kind)]
df

,relation,target_kind,bind_method,conf,n,in_range
0,extends,enum_variant,None,0.5,30,False
1,extends,section,None,0.5,29,False
2,extends,trait,None,0.5,1,True
3,implements,trait,None,0.5,254,True
4,implements,enum_variant,None,0.5,82,False


In [5]:
# Piece 2 evidence — trace ONE out-of-range edge end to end.
# `trait Foo: Send + Sync` captures extends->Send / extends->Sync; std Send/Sync have no
# worktree definition, so a same-named enum_variant / markdown section gets bound instead.
q("""
  SELECT src.entity_name AS subject, src.symbol_kind AS subj_kind,
         src.file_path || ':' || src.line_start AS subject_site,
         e.relation AS predicate,
         e.target_label AS object_label, dst.symbol_kind AS object_kind,
         dst.file_path AS object_defined_in
  FROM edges e
  JOIN nodes src ON src.stable_symbol_id = e.source_stable_id
  JOIN nodes dst ON dst.stable_symbol_id = e.target_stable_id
  WHERE e.relation = 'extends'
    AND dst.symbol_kind IN ('section','enum_variant')
    AND src.entity_name = 'AgentConnection'
  ORDER BY object_label
""")

,subject,subj_kind,subject_site,predicate,object_label,object_kind,object_defined_in
0,AgentConnection,trait,crates/spur-acp/src/connection/mod.rs:74,extends,Send,enum_variant,crates/spur-tui/src/commands/submit_router.rs
1,AgentConnection,trait,crates/spur-acp/src/connection/mod.rs:74,extends,Sync,section,docs/architecture/spur-pm-beads-source-of-trut...


### Why a *correct* resolver still produces a *wrong* graph (the P0 root cause)

This is the subtle, important part of the Tier-0 design. Three facts are simultaneously true:

1. The resolver is **correct**. `resolve_bare_pending_edge` (`extract/tree_sitter.rs:774-791`) range-filters `extends`/`implements` candidates to `relational_target_kinds()`; the enum_variant `Send` and the section `Sync` are filtered out and the edge is left **unresolved**. The Phase-5 fix `f88d9f7b` does exactly this.
2. The fix is **in HEAD**, and both the binary (built 21:53) and the artifact (built 22:12) **post-date** it.
3. The live graph **still has the out-of-range edges** (the table above).

The reconciliation: **edge resolution is persisted per file**, and the incremental build reuses a file's edge bucket whenever its `content_oid` is unchanged. A full re-resolve happens *only* when `manifest_version` changes (`store/build.rs:245`). But:

```
current_manifest_version()  =  SHA256( SCHEMA_VERSION + EXTRACTOR_VERSION + .scm query bytes )
                                                     └── the edge-RESOLUTION logic is NOT in here ──┘
```

`f88d9f7b` changed only `tree_sitter.rs` (Rust resolver code) — no schema, no extractor, no `.scm`. So `manifest_version` is unchanged → incremental builds never re-resolve unchanged trait files → they keep their **pre-fix** out-of-range binds forever. The empty `bind_method` + `conf 0.5` is the fingerprint of the old plain-singleton bind path.

**This is the deepest Tier-0 lesson:** a realization contract that versions *extraction* but not *resolution* lets a landed correctness fix silently fail to reach the graph.

> **✅ Fix landed (P0).** Plan `f25f858a` · task `bd-uioz` folds a `RESOLVER_VERSION` constant into the manifest hash:
> ```
> current_manifest_version() = SHA256( SCHEMA_VERSION + EXTRACTOR_VERSION + RESOLVER_VERSION + .scm bytes )
>                                                                           └── NEW: resolver epoch ──┘
> RESOLVER_VERSION = "2026-06-04-range-constrained-relational-v1"
> ```
> Now any resolution-semantics change (this range fix, the `constructs` split, python-implements) changes `manifest_version` → the next build is a one-time FULL re-resolve that self-heals the persisted graph. Guarded by the unit test `manifest_version_changes_when_resolver_version_changes`. **Bump the constant on every future resolver change.**

**Piece 2's range guard and Piece 5's realization contract are coupled** — a guard you can't force onto the persisted graph isn't really enforced. P0 is the wire that connects them.

## 4. Piece 3 — Predicate sub-axes ⚠️

A predicate is not atomic. Within one `RelationKind` there are **refinement axes** that downstream consumers filter on:

- **`edge_kind`** — a structural sub-type of the predicate. `Calls` splits into `calls` (static), `calls_dyn` (trait-object / dynamic dispatch); `References` splits into `references_hof` (higher-order function passed as a value) vs `references_other`.
- **`bind_method`** — the *resolution provenance* axis (how the object end was chosen): `fqn` (qualified-name exact) ≫ `scope_match` (method resolved within enclosing scope) ≫ `singleton` (only one same-named symbol) ≫ `same_file_duplicate` ≫ `""` (plain/unattributed — the weakest, the one Piece 2's stale edges carry).
- **`confidence_score`** — the scalar trust the above collapses to (`1.0` qualified, `0.5` bare guess, …).

These axes are **partially realized**: `edge_kind` and `confidence` are first-class columns; `bind_method` is populated by the strong paths but left empty by the plain `add_pending_edge` path — so "empty `bind_method`" doubles as a *smell* for under-constrained resolution.

### Sub-axis distribution, and the `calls` polymorphism gap (P1a)

The cell below shows two things at once: (a) how `calls` distributes across `bind_method` (its provenance health), and (b) that `calls` has **no enforced range** — it binds to `field`, `module`, `section`, `constant`, etc., which are not callable. Those are `.into()`/`.join()`/field-access **misresolutions**, the `calls` analogue of Piece 2's problem, and the reason `calls` needs a *terminal* non-callable rejection (P1a). ▼

In [6]:
# Piece 3 evidence (a) — calls provenance health: the bind_method sub-axis.
calls_prov = q("""
  SELECT coalesce(nullif(bind_method,''), '(empty)') AS bind_method,
         edge_kind, count(*) AS n
  FROM edges WHERE relation='calls'
  GROUP BY 1,2 ORDER BY n DESC""")

# Piece 3 evidence (b) — calls has NO enforced range: object kinds that are not callable.
calls_range = q("""
  SELECT dst.symbol_kind AS object_kind, count(*) AS n,
         (dst.symbol_kind NOT IN ('function','method')) AS out_of_domain
  FROM edges e JOIN nodes dst ON dst.stable_symbol_id=e.target_stable_id
  WHERE e.relation='calls'
  GROUP BY 1 ORDER BY n DESC""")

print("calls — provenance (bind_method × edge_kind):"); print(calls_prov.to_string(index=False))
print("\ncalls — object-kind range (function/method = legit, rest = misresolution):")
print(calls_range.to_string(index=False))

calls — provenance (bind_method × edge_kind):
         bind_method edge_kind     n
           singleton     calls 10320
         scope_match     calls  6452
             (empty)     calls  6024
macro_body_singleton     calls  1777
             (empty) calls_dyn    92
                 fqn     calls    70

calls — object-kind range (function/method = legit, rest = misresolution):
object_kind     n  out_of_domain
   function 11660          False
     method  8286          False
      field  4709           True
     module    40           True
    section    30           True
   constant     7           True
       enum     2           True
      macro     1           True


## 5. Piece 4 — Inverses / cardinality / transitivity ❌ (P2)

A mature ontology declares the **algebra** of each predicate — the properties that let a reasoner derive facts that were never explicitly extracted:

| Property | Meaning | Examples |
|---|---|---|
| **Inverse** | `p(a,b) ⇒ p⁻¹(b,a)` | `calls⁻¹ = called_by`, `contains⁻¹ = contained_by`, `imports⁻¹ = imported_by` |
| **Cardinality** | how many objects a subject may have | `defines` 1→many; `implements` many→many; `extends` (Rust supertrait) many→many |
| **Transitivity** | `p(a,b) ∧ p(b,c) ⇒ p(a,c)` | `contains` transitive (file ⊇ impl ⊇ method); `extends` transitive (supertrait chain) |
| **Symmetry / reflexivity** | rarely needed here, but part of the vocabulary | — |

**Why it matters:** without declared inverses, "who calls X?" requires a full reverse scan instead of a derived `called_by` view; without declared transitivity, "is X reachable under `contains`?" can't be answered by the T-Box and must be recomputed per query. This is exactly the metadata Tier 3 (inference) and Tier 4 (entailment/governance) would consume.

**Current state: entirely absent.** `RelationKind` (`schema.rs:282`) is a plain enum with no associated algebra; `GraphEdge` (`schema.rs:203`) carries only `directed: bool`. The cell below confirms the *shape* — `directed` is the only relational-algebra bit the persisted edge has, and every edge sets it. There is nowhere a consumer can ask "what is `calls`'s inverse?" ▼

In [7]:
# Piece 4 evidence — the persisted edge schema's only algebra bit is `directed`.
# There is NO inverse / cardinality / transitivity column to query — confirm what columns exist.
cols = q("""SELECT column_name, data_type
            FROM information_schema.columns
            WHERE table_schema='main' AND table_name='edges'
            ORDER BY ordinal_position""")
print("edges columns (note: no inverse/cardinality/transitivity anywhere):")
print(cols.to_string(index=False))

# Empirically, every predicate is stored as a flat directed edge — the algebra must be
# re-derived per query today. Example: 'who calls X' has to scan, since calls^-1 is undeclared.
print("\nPredicates present in the graph (each a bare directed edge, no declared algebra):")
print(q("SELECT relation, count(*) n FROM edges GROUP BY 1 ORDER BY n DESC").to_string(index=False))

edges columns (note: no inverse/cardinality/transitivity anywhere):
     column_name data_type
source_stable_id   VARCHAR
target_stable_id   VARCHAR
          src_id    BIGINT
          dst_id    BIGINT
    target_label   VARCHAR
        relation   VARCHAR
      confidence   VARCHAR
confidence_score     FLOAT
       edge_kind   VARCHAR
     bind_method   VARCHAR

Predicates present in the graph (each a bare directed edge, no declared algebra):
  relation     n
  contains 49890
     calls 24735
   defines 18512
   imports  4979
constructs   951
implements   336
references   110
   extends    60
     links    19


## 6. Piece 5 — Versioned realization contract ⚠️

The previous four pieces define what a fact *means*. Piece 5 is the **governance layer**: which predicates each language actually *realizes*, and the machinery that keeps the declared contract and the running code from drifting apart.

### 5a. The Relation Coverage Matrix (`queries/README.md`)

`Y` = realized · `—` = not realizable for that family · `TODO` = known gap (must be reviewed, never silent):

| Predicate | Rust | Python | TypeScript | Tsx | Cpp | Markdown |
|---|---|---|---|---|---|---|
| imports | Y | Y | Y | Y | Y | Y(links) |
| calls | Y | Y | Y | Y | Y | — |
| constructs | Y | Y | Y | Y | Y | — |
| contains | Y | Y | Y | Y | Y | Y |
| defines | Y | Y | Y | Y | Y | — |
| references (HOF) | Y | Y | TODO | TODO | Y | — |
| links | — | — | — | — | — | Y |
| implements | Y | Y | Y | Y | — | — |
| extends | Y | Y | Y | Y | Y | — |

A predicate is realized for a language by adding the matching `@capture` to that language's `spur-edges.scm`; `emit_edges` (`languages.rs`) dispatches capture-name → `RelationKind`. The matrix is the human-readable contract.

### 5b. Two enforcement gaps

**P1b — the gate enforces *definitions*, not *predicates*.** The contract test in `extract/languages.rs` checks that every `@definition.*` capture has a `definition_kind_map` key (Piece 1 integrity), that `tags` queries are non-empty, etc. It asserts **nothing** about the relation matrix above — a language could quietly stop realizing `implements` and no test would fail. The matrix can drift from reality silently. *Fix:* a relation-coverage contract test derived from the matrix, with `TODO`s as explicit allow-listed gaps.

**P0 — the cache key versions *extraction*, not *resolution*.** As shown in Piece 2, `manifest_version` hashes schema + extractor + `.scm` bytes but not the resolver logic, so a resolver fix never invalidates the incremental edge cache. *Fix:* fold a `RESOLVER_VERSION` constant into `manifest_version` (plan `f25f858a`).

### Why this is the keystone piece

Pieces 1-4 are *claims about the graph*. Piece 5 is *the only piece that makes the other four enforceable on the persisted artifact*. Without a versioned realization contract: a domain/range guard (Piece 2) can be bypassed by a stale cache; a class-coverage rule (Piece 1) is enforced but its relational sibling (Piece 5a) is not; and any algebra you add (Piece 4) has no drift guard. **Tier-0 is only as strong as Piece 5.**

## 7. The resolution pipeline — where each Tier-0 guard lives

A triple is born unresolved (a *label*) and earns its object through resolution. Tracing the path shows exactly where each Tier-0 constraint is (or should be) enforced:

```
 ┌─ EXTRACT ────────────────────────────────────────────────────────────────┐
 │ tree-sitter parse → @captures                                             │
 │   @definition.<kind>  ─→  GraphNode (Piece 1: class assigned here)        │
 │   @import/@call/@implements/@extends/@reference.name ─→ emit_edges()      │
 │        (languages.rs)  capture-name → RelationKind  (Piece 5a realizes)   │
 │   produces a PendingEdge { source, target_NAME, relation }  (object = a   │
 │   string label, not yet a node)                                           │
 └──────────────────────────────┬───────────────────────────────────────────┘
                                 │  per-file facts persisted to a bucket
 ┌─ RESOLVE  (resolve_pending_edges, tree_sitter.rs:464) ────────────────────┐
 │ dispatch by relation:                                                     │
 │   CallsDyn   → trait-method candidates (edge_kind sub-axis, Piece 3)      │
 │   References → HOF allowlist (Piece 5a) + callable-kind filter            │
 │   Calls      → qualified candidates → Constructs reclassification (Piece2)│
 │               └─ MISS → callable filter … but NOT terminal (P1a leak)     │
 │   Imports    → import candidate filter                                    │
 │   else (implements/extends/…) → resolve_bare_pending_edge                 │
 │        └─ relational_target_kinds() RANGE GUARD (Piece 2)  ── line 774    │
 │             in-range singleton → bind (provenance set, Piece 3)           │
 │             out-of-range / ambiguous → leave UNRESOLVED (honest label)    │
 └──────────────────────────────┬───────────────────────────────────────────┘
                                 │  resolved edges written back into the bucket
 ┌─ PERSIST  (store/build.rs) ───────────────────────────────────────────────┐
 │ artifact_from_facts[_incremental]                                         │
 │   incremental: reuse bucket when content_oid unchanged …                  │
 │   full re-resolve ONLY when manifest_version changed (Piece 5b / P0)      │
 └───────────────────────────────────────────────────────────────────────────┘
```

**The cross-cutting insight:** the guards live in RESOLVE, but their *effect on the persisted graph* is gated by PERSIST's cache key. Piece 2's range guard is real code on the RESOLVE line, but Piece 5b's versioning bug means it doesn't reach the artifact for unchanged files. **Every Tier-0 guarantee is the conjunction of a guard *and* a cache-invalidation trigger.** That coupling is the single most important thing to internalize from this design.

### Data architecture (mermaid)

The same EXTRACT → RESOLVE → PERSIST flow as a diagram, with the **Tier-0 T-Box as the constraint layer** wired into the stages it governs. Dashed red = the two open Tier-0 gaps (P0: `manifest_version` doesn't version the resolver; P2: predicate algebra absent).

```mermaid
flowchart TB
  subgraph SRC[" Source worktree "]
    FILES["*.rs · *.py · *.ts · *.cpp · *.md"]
  end

  subgraph EXTRACT["① EXTRACT — tree-sitter (extract/languages.rs)"]
    PARSE["parse → @captures"]
    NODES["GraphNode<br/>(class assigned)"]
    PEND["PendingEdge<br/>{subject, target_LABEL, relation}"]
  end

  subgraph TBOX["Tier-0 T-Box — the constraint layer"]
    direction LR
    P1["Piece 1<br/>Classes (NodeKind)"]
    P2["Piece 2<br/>domain / range"]
    P3["Piece 3<br/>sub-axes<br/>edge_kind·bind_method·conf"]
    P4["Piece 4<br/>algebra<br/>inverse·card·transitivity"]
    P5["Piece 5<br/>realization contract<br/>+ version"]
  end

  subgraph RESOLVE["② RESOLVE — resolve_pending_edges (tree_sitter.rs:464)"]
    DISP["dispatch by relation"]
    RANGE{"range guard<br/>relational_target_kinds()"}
    BIND["bind object →<br/>stamp confidence + bind_method"]
    UNRES["out-of-range / ambiguous<br/>→ leave UNRESOLVED (honest label)"]
  end

  subgraph PERSIST["③ PERSIST — store/build.rs"]
    BUCKET["per-file edge bucket"]
    MANIFEST{{"manifest_version<br/>cache key"}}
    PARQUET[(".spur/graph CURRENT<br/>content-addressed Parquet")]
    DUCK[(".spur/analyst.duckdb")]
  end

  subgraph CONSUME[" Consumers "]
    CODEMCP["code_* MCP tools"]
    ANALYST["spur-analyst<br/>DuckPGQ · Onager"]
    LADDER["Tier 1→4 ladder"]
  end

  FILES --> PARSE
  PARSE --> NODES
  PARSE --> PEND
  P1 -. types .-> NODES
  PEND --> DISP --> RANGE
  P2 -. enforces .-> RANGE
  RANGE -->|in range| BIND
  RANGE -->|out of range| UNRES
  P3 -. stamps .-> BIND
  BIND --> BUCKET
  UNRES --> BUCKET
  BUCKET --> PARQUET
  MANIFEST -. gates full re-resolve .-> BUCKET
  P5 -. "P0: does NOT version resolver" .-> MANIFEST
  PARQUET --> DUCK
  PARQUET --> CODEMCP
  DUCK --> ANALYST
  PARQUET --> LADDER
  P4 -. "P2: absent → blocks" .-> LADDER

  classDef gap stroke:#c0392b,stroke-width:2px,stroke-dasharray:4 3;
  class P4,P5 gap;
```

And the **T-Box schema** itself — the triple, with each Tier-0 piece annotated onto the field it governs:

```mermaid
classDiagram
  direction LR
  class GraphNode {
    +stable_symbol_id : identity
    +NodeKind kind  «Piece 1»
    +qualified_name
    +file_path
  }
  class GraphEdge {
    +RelationKind relation  «predicate»
    +edge_kind        «Piece 3»
    +bind_method      «Piece 3»
    +confidence_score «Piece 3»
    +directed : bool
  }
  class RelationKind {
    <<enumeration>>
    Imports Calls Constructs
    Contains Implements Defines
    References Extends Links Touches
  }
  GraphNode "1  subject (domain «Piece 2»)" --> "0..*" GraphEdge
  GraphEdge "0..*" --> "1  object (range «Piece 2»)" GraphNode
  GraphEdge --> RelationKind
  note for RelationKind "Piece 4 algebra (inverse / cardinality / transitivity) attaches HERE — currently absent (P2)"
```

## 8. The predicate ladder — Tier 0 → 1 → 2 → 3 → 4

Tier 0 is the foundation; each higher tier consumes the tier below. The ladder is *additive* — nothing above is trustworthy if Tier 0 leaks.

| Tier | Name | What it adds | Input it needs from below | Status |
|---|---|---|---|---|
| **0** | **T-Box** (this notebook) | Classes, predicate domain/range, sub-axes, algebra, realization contract | tree-sitter captures | the focus here — pieces 1✅ 2-3-5⚠️ 4❌ |
| **1** | **Disambiguate latent AST** | Turn ambiguous captures into precise predicates: `calls→constructs` split, `extends→implements` (Python Protocol/ABC), range-constrained binds | Tier-0 classes + range signatures | landed (constructs, python-implements, range guard) |
| **2** | **Cross-artifact fusion** | Merge facts about the *same* `stable_symbol_id` across rebuilds/branches/temporal snapshots; reconcile confidence | Tier-0 identity (`stable_symbol_id`) + provenance | partial (temporal_edges, symbol_snapshots exist) |
| **3** | **Inference / embeddings / confidence** | Derive *unstated* edges: transitive reachability, `called_by` views, semantic similarity, probabilistic links | Tier-0 **algebra** (Piece 4) + Tier-2 fused facts | blocked on Piece 4 |
| **4** | **Entailment / governance** | Rules over the graph: "a `pub` trait method with zero resolved callers is dead", policy/lint entailments, contract checks | Tier-0 realization contract (Piece 5) + Tier-3 inference | blocked on Pieces 4 & 5 |

**Reading the dependency:** Tier 3's "who calls X" view *is* `calls⁻¹`, which needs Piece 4's inverse declaration. Tier 4's governance rules need Piece 5's contract to know which predicates are even *expected* to be present per language (else "zero callers" can't distinguish "truly dead" from "this language doesn't realize that predicate"). This is why **completing Tier-0 Pieces 4 and 5 is the unlock for everything above** — and why P0/P2 are prioritized over cosmetic gaps.

## 9. Open gaps & roadmap

| ID | Tier-0 piece | Gap | Severity | Fix | Status |
|---|---|---|---|---|---|
| **P0** | 5b + 2 | `manifest_version` doesn't hash resolver semantics → landed range fix never reaches the persisted graph; live graph violates implements/extends domain/range | **highest** (silent correctness regression) | add `RESOLVER_VERSION` to `manifest_version` → forces one full rebuild that self-heals | ✅ **implemented** in `store/build.rs` (+29/−6) — `RESOLVER_VERSION = "2026-06-04-range-constrained-relational-v1"` folded into the hash, new `manifest_version_changes_when_resolver_version_changes` test; plan `f25f858a` · task `bd-uioz` (awaiting review → integrate, then rebuild) |
| **P1a** | 2 | `calls` resolution non-terminal → binds to `field`/`module`/`section` (4.7k field misresolutions) | high (live resolver bug) | make `calls` terminally reject non-callable singletons; emit `constructs` for type targets | queued |
| **P1b** | 5a | realization gate enforces only definitions, not the relation matrix → predicate coverage can drift silently | medium (structural) | relation-coverage contract test derived from `queries/README.md` | queued |
| **P2** | 4 | no inverse / cardinality / transitivity metadata on `RelationKind` | medium (blocks Tier 3-4) | per-relation ontology metadata table | queued |

### Sequencing rationale
1. **P0 first** — cheapest change, and a full rebuild afterwards makes every *other* finding measurable on a clean graph (today's numbers are partly contaminated by the stale cache). **← landed in code; the bump now forces that rebuild automatically.**
2. **P1a** — the actual `calls` resolver defect, independent of P0.
3. **P1b / P2** — declare the contract (5a) and the algebra (4) as first-class, enforced data; these unlock Tiers 3-4.

### How to re-run this spec
Every evidence cell is a live read-only query against `.spur/analyst.duckdb`. With P0 now implemented, the **next** graph build sees a changed `manifest_version` → does a FULL re-resolve → self-heals. After that rebuild, re-run the notebook: Piece 2's out-of-range rows should drop to **0**, and the `extends → trait` / `implements → trait` counts should rise correspondingly. That diff is the acceptance test for P0 at the *graph* level (the unit test guards it at the *code* level — see §3).

---
*Generated against artifact `a66543657…` (pre-P0-rebuild). Re-execute top-to-bottom after the next build to confirm the violations clear.*